## Business questions — solve in order:

- Add a revenue column to orders
- Convert order_date to datetime and extract month
- Filter completed orders only
- Merge with customers to get name and region
- Total revenue per region
- Top spending customer by name
- Monthly revenue trend — which month was best?
- Most popular product by quantity sold

In [2]:
import pandas as pd

orders = pd.DataFrame({
    "order_id":   [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "customer_id":[101, 102, 101, 103, 102, 104, 103, 101, 105, 104, 102, 103],
    "product":    ["Laptop", "Phone", "Phone", "Laptop", "Tablet", "Phone",
                   "Tablet", "Laptop", "Phone", "Laptop", "Phone", "Tablet"],
    "quantity":   [1, 2, 1, 2, 1, 3, 2, 1, 2, 1, 1, 3],
    "unit_price": [1200, 800, 800, 1200, 450, 800, 450, 1200, 800, 1200, 800, 450],
    "order_date": ["2024-01-10", "2024-01-15", "2024-02-01", "2024-02-14",
                   "2024-02-20", "2024-03-01", "2024-03-10", "2024-03-15",
                   "2024-03-20", "2024-04-01", "2024-04-10", "2024-04-15"],
    "status":     ["completed", "completed", "completed", "cancelled", "completed",
                   "completed", "completed", "cancelled", "completed", "completed",
                   "completed", "completed"]
})

customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "name":        ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "region":      ["North", "South", "North", "East", "South"]
})

In [3]:
# - Add a revenue column to orders
revenue = orders["quantity"] * orders["unit_price"]
orders["revenue"] = revenue
orders.head()

,order_id,customer_id,product,quantity,unit_price,order_date,status,revenue
0,1,101,Laptop,1,1200,2024-01-10,completed,1200
1,2,102,Phone,2,800,2024-01-15,completed,1600
2,3,101,Phone,1,800,2024-02-01,completed,800
3,4,103,Laptop,2,1200,2024-02-14,cancelled,2400
4,5,102,Tablet,1,450,2024-02-20,completed,450


In [4]:
#- Convert order_date to datetime and extract month

orders["order_date"] = pd.to_datetime(orders["order_date"], format = "%Y-%m-%d")
orders["month"]=orders["order_date"].dt.month
orders.head()

,order_id,customer_id,product,quantity,unit_price,order_date,status,revenue,month
0,1,101,Laptop,1,1200,2024-01-10,completed,1200,1
1,2,102,Phone,2,800,2024-01-15,completed,1600,1
2,3,101,Phone,1,800,2024-02-01,completed,800,2
3,4,103,Laptop,2,1200,2024-02-14,cancelled,2400,2
4,5,102,Tablet,1,450,2024-02-20,completed,450,2


In [5]:
#. - Filter completed orders only

completed_orders = orders[orders["status"] == "completed"]
completed_orders

,order_id,customer_id,product,quantity,unit_price,order_date,status,revenue,month
0,1,101,Laptop,1,1200,2024-01-10,completed,1200,1
1,2,102,Phone,2,800,2024-01-15,completed,1600,1
2,3,101,Phone,1,800,2024-02-01,completed,800,2
4,5,102,Tablet,1,450,2024-02-20,completed,450,2
5,6,104,Phone,3,800,2024-03-01,completed,2400,3
6,7,103,Tablet,2,450,2024-03-10,completed,900,3
8,9,105,Phone,2,800,2024-03-20,completed,1600,3
9,10,104,Laptop,1,1200,2024-04-01,completed,1200,4
10,11,102,Phone,1,800,2024-04-10,completed,800,4
11,12,103,Tablet,3,450,2024-04-15,completed,1350,4


In [9]:
# - Merge with customers to get name and region
merge_order = pd.merge(completed_orders, customers, on = "customer_id", how = "left")
merge_order

,order_id,customer_id,product,quantity,unit_price,order_date,status,revenue,month,name,region
0,1,101,Laptop,1,1200,2024-01-10,completed,1200,1,Alice,North
1,2,102,Phone,2,800,2024-01-15,completed,1600,1,Bob,South
2,3,101,Phone,1,800,2024-02-01,completed,800,2,Alice,North
3,5,102,Tablet,1,450,2024-02-20,completed,450,2,Bob,South
4,6,104,Phone,3,800,2024-03-01,completed,2400,3,Diana,East
5,7,103,Tablet,2,450,2024-03-10,completed,900,3,Charlie,North
6,9,105,Phone,2,800,2024-03-20,completed,1600,3,Eve,South
7,10,104,Laptop,1,1200,2024-04-01,completed,1200,4,Diana,East
8,11,102,Phone,1,800,2024-04-10,completed,800,4,Bob,South
9,12,103,Tablet,3,450,2024-04-15,completed,1350,4,Charlie,North


In [11]:
# - Total revenue per region
merge_order.groupby("region")["revenue"].sum()

region
East     3600
North    4250
South    4450
Name: revenue, dtype: int64

In [14]:
# - Top spending customer by name
customer_spending = merge_order.groupby("name")["revenue"].sum()
customer_spending


name
Alice      2000
Bob        2850
Charlie    2250
Diana      3600
Eve        1600
Name: revenue, dtype: int64

In [15]:
customer_spending.idxmax()

'Diana'

In [17]:
# - Monthly revenue trend — which month was best?
best_month = merge_order.groupby("month")["revenue"].sum()
best_month.idxmax()

np.int32(3)

In [21]:
# - Most popular product by quantity sold

popular_product= merge_order.groupby("product")["quantity"].sum()
popular_product

product
Laptop    2
Phone     9
Tablet    6
Name: quantity, dtype: int64

In [22]:
popular_product.idxmax()

'Phone'